# IndiKit domain scrape & comparison

Scrapes the indicator library of every IndiKit domain listed in `files/links` (one URL per line) using `files/scrape_indikit.py`. Domains that already have `data/{domain}/indicators.json` are skipped — delete that folder to re-scrape.

Discovery goes via `sitemap.xml` (fallback: crawling sector pages), then each indicator page is parsed for title, sector, metadata tables and narrative sections. Output lands in `data/{domain}/indicators.json` + `.csv`. The scraper waits 1 s between requests, so expect several minutes per domain.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("files").resolve()))
import scrape_indikit

# One URL per line in files/links; strip trailing slashes so domain folder
# names come out clean (e.g. www_indikit_net, not www_indikit_net/).
DOMAINS = [
    line.strip().rstrip("/")
    for line in Path("files/links").read_text().splitlines()
    if line.strip()
]
DOMAINS

['https://indicators.savethechildren.net',
 'https://winrock.indikit.net',
 'https://whh.indikit.net',
 'https://helvetas.indikit.net',
 'https://tearfund.indikit.net',
 'https://adra.indikit.net',
 'https://irw.indikit.net',
 'https://tdh.indikit.net',
 'https://www.indikit.net',
 'https://zoa.indikit.net']

In [2]:
for base_url in DOMAINS:
    domain = base_url.split("//")[1].replace(".", "_")
    if (Path("data") / domain / "indicators.json").exists():
        print(f"=== {base_url} === already scraped — delete data/{domain}/ to re-scrape")
        continue
    print(f"=== {base_url} ===")
    scrape_indikit.main(base_url)
    print()

=== https://indicators.savethechildren.net ===
[1/3] Discovering sectors on https://indicators.savethechildren.net
  30 sectors found
[2/3] Discovering indicators
  sitemap found: 180 indicator URLs
[3/3] Scraping 180 indicator pages
  [1/180] https://indicators.savethechildren.net/indicator/207-health-outside-strategic-sub-themes-other/199-of-individuals-who-received-health-related-kits
  [2/180] https://indicators.savethechildren.net/indicator/31-monitoring-demanding-child-rights-with-children/131-gender-equality-coordination
  [3/180] https://indicators.savethechildren.net/indicator/31-monitoring-demanding-child-rights-with-children/187-of-recommendations-by-human-rights-mechanisms-accepted-endorsed-by-the-government
  [4/180] https://indicators.savethechildren.net/indicator/31-monitoring-demanding-child-rights-with-children/195-of-sector-specific-action-plans-at-national-or-subnational-levels-costed-as-a-result-of-influence
  [5/180] https://indicators.savethechildren.net/indicator

In [3]:
import pandas as pd

frames = {}
for base_url in DOMAINS:
    domain = base_url.split("//")[1].replace(".", "_")
    df = pd.read_json(f"data/{domain}/indicators.json")
    frames[domain] = df
    print(
        f"{domain}: {len(df)} indicators | "
        f"{df['sector'].notna().sum()} with sector | "
        f"{df.shape[1]} fields captured"
    )

pd.concat(
    [df["sector"].value_counts().rename(domain) for domain, df in frames.items()],
    axis=1,
).fillna(0).astype(int)

indicators_savethechildren_net: 179 indicators | 179 with sector | 38 fields captured
winrock_indikit_net: 42 indicators | 42 with sector | 12 fields captured
whh_indikit_net: 122 indicators | 122 with sector | 12 fields captured
helvetas_indikit_net: 87 indicators | 87 with sector | 28 fields captured
tearfund_indikit_net: 230 indicators | 230 with sector | 11 fields captured
adra_indikit_net: 507 indicators | 507 with sector | 13 fields captured
irw_indikit_net: 480 indicators | 480 with sector | 12 fields captured
tdh_indikit_net: 65 indicators | 65 with sector | 23 fields captured
www_indikit_net: 518 indicators | 518 with sector | 14 fields captured
zoa_indikit_net: 303 indicators | 303 with sector | 12 fields captured


,indicators_savethechildren_net,winrock_indikit_net,whh_indikit_net,helvetas_indikit_net,tearfund_indikit_net,adra_indikit_net,irw_indikit_net,tdh_indikit_net,www_indikit_net,zoa_indikit_net
sector,,,,,,,,,,
Other Strategy Indicators,50,0,0,0,0,0,0,0,0,0
Output Indicators,49,0,0,0,0,0,0,0,0,0
"Institutions with basic drinking water, limited sanitation or hygiene services",12,0,0,0,0,0,0,0,0,0
"Access to at least basic drinking water, basic hygiene or limited (humanitarian contexts) or basic (development contexts) sanitation",10,0,0,0,0,0,0,0,0,0
Children Achieving Foundational level in Literacy,9,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
Peacebuilding - General,0,0,0,0,0,0,0,0,0,4
Gender-Based Violence,0,0,0,0,0,0,0,0,0,4
Social Inclusion and Protection,0,0,0,0,0,0,0,0,0,3


In [4]:
frames["zoa_indikit_net"][["title", "sector", "url"]].head(10)

,title,sector,url
0,Individual Dietary Diversity Score,Food Security,https://zoa.indikit.net/indicator/5083-individ...
1,Household Dietary Diversity Score,Food Security,https://zoa.indikit.net/indicator/5089-househo...
2,FSL: Food Consumption Score,Sector specific results of CVA,https://zoa.indikit.net/indicator/5090-fsl-foo...
3,FSL: Coping Strategy Index,Sector specific results of CVA,https://zoa.indikit.net/indicator/5091-fsl-cop...
4,Months of Adequate Household Food Provisioning,Food Security,https://zoa.indikit.net/indicator/5092-months-...
5,Awareness of Appropriate Diet,Food Security,https://zoa.indikit.net/indicator/5099-awarene...
6,Prevalence of Diarrhoea Among Children,Hygiene,https://zoa.indikit.net/indicator/5114-prevale...
7,Appropriate Treatment of Diarrhoea - ORS,Hygiene,https://zoa.indikit.net/indicator/5117-appropr...
8,Appropriate Treatment of Diarrhoea - Fluids,Hygiene,https://zoa.indikit.net/indicator/5118-appropr...
9,Appropriate Treatment of Diarrhoea - Food,Hygiene,https://zoa.indikit.net/indicator/5119-appropr...


## Domain comparison

Runs `files/compare_domains.py` on the two scraped datasets. It writes four CSVs to `comparison/`:

- `overview.csv` — indicator counts per sector per domain
- `shared_indicators.csv` — fuzzy title matches across domains (Jaccard word overlap ≥ 0.6)
- `unique_<domain>.csv` — indicators with no fuzzy match in the other domain
- `field_coverage.csv` — how completely each domain fills each metadata field

Caveats to keep in mind when reading the results: the fuzzy title match is a crude Jaccard word overlap (near-identical wording with different subjects scores high, rephrased-but-identical indicators score low), matches are not one-to-one, and the two domains use completely different sector taxonomies and page templates — so sector and field comparisons mostly show *structural* differences between the sites, not content overlap.

In [5]:
import compare_domains

compare_domains.main([
    f"data/{base_url.split('//')[1].replace('.', '_')}/indicators.json"
    for base_url in DOMAINS
])

Sector coverage:
 domain                                              adra.indikit.net  \
sector                                                                 
ACCESS TO FOOD INDICATORS                                        5.0   
Access and Inclusion Indicators                                  0.0   
Access and Learning Environment                                  9.0   
Access to Water Indicators                                       0.0   
Access to at least basic drinking water, basic ...               0.0   
...                                                              ...   
Weather or climate information services coverage                 0.0   
Wellbeing & Mental Health of C&Y and Caregivers                  0.0   
Women receiving postnatal health check within 2...               0.0   
Youth Civic Engagement Indicators                                0.0   
review other indicators                                          1.0   

domain                                       

In [6]:
# Shared indicators — fuzzy title matches across the two domains, best matches first
shared = pd.read_csv("comparison/shared_indicators.csv")
shared.sort_values("similarity", ascending=False).head(15)

,domain_a,title_a,sector_a,domain_b,title_b,sector_b,similarity
2191,www.indikit.net,Household Expenditure,INCOME AND ASSETS-RELATED INDICATORS,zoa.indikit.net,Household Expenditure,Income and Employment,1.0
2190,www.indikit.net,Household Average Income,INCOME AND ASSETS-RELATED INDICATORS,zoa.indikit.net,Household Average Income,Income and Employment,1.0
2189,www.indikit.net,Number of Products Sold,Market Supply-Related Indicators,zoa.indikit.net,Number of Products Sold,Income and Employment,1.0
2188,www.indikit.net,Number of Clients Served,Market Supply-Related Indicators,zoa.indikit.net,Number of Clients Served,Market Development,1.0
2186,www.indikit.net,Quality of Provided Services,Agricultural Extension Indicators,zoa.indikit.net,Quality of Provided Services,Market Development,1.0
2185,www.indikit.net,Diversity of Products Sold,Market Supply-Related Indicators,zoa.indikit.net,Diversity of Products Sold,Market Development,1.0
5,indicators.savethechildren.net,Gender equitable norms,Other Strategy Indicators,tearfund.indikit.net,Gender Equitable Norms,Gender-based violence (GBV) and psychosocial s...,1.0
2184,www.indikit.net,Use of Promoted Products / Services,Agricultural Extension Indicators,zoa.indikit.net,Use of Promoted Products / Services,Market Development,1.0
2144,www.indikit.net,People Benefiting,Results of Advocacy Strategies,zoa.indikit.net,People Benefiting,"Community Engagement, Participation & Advocacy",1.0
2145,www.indikit.net,Adopted Recommendations,Results of Advocacy Strategies,zoa.indikit.net,Adopted Recommendations,"Community Engagement, Participation & Advocacy",1.0


In [7]:
# Field coverage — % of indicators where each metadata field is filled, per domain.
# Mostly reflects the two sites' different page templates: tdh uses metadata
# tables + "Purpose"/"Definition" sections, zoa uses "Indicator's Wording" etc.
field_cov = pd.read_csv("comparison/field_coverage.csv")
field_cov.pivot_table(index="field", columns="domain", values="pct", fill_value=0).sort_values(
    list(field_cov["domain"].unique()), ascending=False
)

domain,adra.indikit.net,helvetas.indikit.net,indicators.savethechildren.net,irw.indikit.net,tdh.indikit.net,tearfund.indikit.net,whh.indikit.net,winrock.indikit.net,www.indikit.net,zoa.indikit.net
field,,,,,,,,,,
domain,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
sector,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
slug,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
title,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
url,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
...,...,...,...,...,...,...,...,...,...,...
Important considerations,0.0,0.0,0.0,0.0,43.1,0.0,0.0,0.0,0.0,0.0
Example of survey questions,0.0,0.0,0.0,0.0,29.2,0.0,0.0,0.0,0.0,0.0
section:Have a minute to help IndiKit?,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0


In [ ]:
# Indicators unique to each domain (no fuzzy match ≥0.6 in any other)
for base_url in DOMAINS:
    domain = base_url.split("//")[1].replace(".", "_")
    unique = pd.read_csv(f"comparison/unique_{domain}.csv")
    print(f"{domain}: {len(unique)} unique indicators")

pd.read_csv("comparison/unique_zoa_indikit_net.csv").head(10)

## Semantic comparison (embeddings)

Title-based fuzzy matching misses indicators that are worded differently across sites. Here we compare at the *meaning* level instead:

1. **Canonical text** — per indicator, concatenate the meaning-bearing fields. The domains use different templates, so fields are selected by name pattern (wording / phrasing / purpose / definition / full statement) rather than a fixed list.
2. **Local embeddings** — `sentence-transformers` (`all-MiniLM-L6-v2`), free, runs locally (`pip install sentence-transformers`; first run downloads the ~90 MB model).
3. **Similarity matrix** — cosine similarity between ZOA's indicators and those of every other scraped domain.
4. **Best-match table** — for each ZOA indicator, its closest counterpart anywhere in the other libraries; mutual best matches are the strongest "probably the same indicator" candidates.
5. **Search** — embed a free-text query and retrieve the nearest indicators from all libraries.

Caveat: embeddings measure *topical* similarity, not measurement equivalence — "% of children enrolled" vs "# of schools built" still score moderately high because both are about education. Treat high-scoring pairs as candidates to eyeball, not as confirmed matches.

In [13]:
# 1. Canonical text per indicator — meaning-bearing fields only.
# Every IndiKit domain names its sections differently ("Indicator’s Wording",
# "Indicator Phrasing", "Indicator full statement", "Definition", ...), so
# select fields by pattern instead of a hardcoded per-domain map.
SEMANTIC_PATTERNS = ("wording", "phrasing", "purpose", "definition", "full statement")


def canonical_text(row):
    parts = [str(row["title"])]
    for field, v in row.items():
        if not any(p in field.lower() for p in SEMANTIC_PATTERNS):
            continue
        if isinstance(v, str) and v.strip():
            parts.append(v.strip())
    return "\n".join(parts)


corpus = pd.DataFrame(
    {
        "domain": domain,
        "title": row["title"],
        "sector": row["sector"],
        "url": row["url"],
        "text": canonical_text(row),
    }
    for domain, df in frames.items()
    for _, row in df.iterrows()
)
print(corpus.groupby("domain")["text"].agg(n="count", avg_chars=lambda s: s.str.len().mean().round()))
print("\nExample canonical text:\n", corpus["text"].iloc[0][:300])

                                  n  avg_chars
domain                                        
adra_indikit_net                507      432.0
helvetas_indikit_net             87     2504.0
indicators_savethechildren_net  179     1554.0
irw_indikit_net                 480      564.0
tdh_indikit_net                  65     2199.0
tearfund_indikit_net            230      557.0
whh_indikit_net                 122      724.0
winrock_indikit_net              42      562.0
www_indikit_net                 518      457.0
zoa_indikit_net                 303      521.0

Example canonical text:
 # of individuals who received health-related kits
This indicator measures the total number of unique individuals who have received health-related kits or in-kind support (e.g., hygiene kits, maternal and newborn kits, personal protective equipment (PPE), medical supplies) distributed by Save the Chi


In [14]:
# 2. Local embeddings — normalized so dot product = cosine similarity.
# Note: the model truncates input at 256 tokens, so the embedding mainly
# reflects the title + the first ~1000 characters of the canonical text.
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(
    corpus["text"].tolist(), normalize_embeddings=True, show_progress_bar=False
)
print(embeddings.shape)

/Users/hiddemakimei/Desktop/git/vu/ML4QS/code/.venv/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


(2533, 384)


In [17]:
# 3+4. Similarity between the reference domain (zoa) and all other domains.
# For each zoa indicator: its best match anywhere in the other libraries.
# "mutual" = that indicator's best zoa match is this one — strongest duplicates.
import numpy as np

REFERENCE = "zoa_indikit_net"
is_ref = (corpus["domain"] == REFERENCE).to_numpy()
ref_meta = corpus[is_ref].reset_index(drop=True)
other_meta = corpus[~is_ref].reset_index(drop=True)

sim = embeddings[is_ref] @ embeddings[~is_ref].T  # n_zoa × n_others
best_other = sim.argmax(axis=1)
best_ref = sim.argmax(axis=0)

matches = pd.DataFrame(
    {
        "zoa_title": ref_meta.loc[i, "title"],
        "match_title": other_meta.loc[j, "title"],
        "match_domain": other_meta.loc[j, "domain"],
        "similarity": round(float(sim[i, j]), 3),
        "mutual": bool(best_ref[j] == i),
        "zoa_sector": ref_meta.loc[i, "sector"],
        "match_sector": other_meta.loc[j, "sector"],
    }
    for i, j in enumerate(best_other)
).sort_values("similarity", ascending=False).reset_index(drop=True)

matches.to_csv("comparison/semantic_matches.csv", index=False)
print(f"mutual top-1 pairs: {matches['mutual'].sum()} of {len(matches)}")
matches.head(20)

mutual top-1 pairs: 248 of 303


,zoa_title,match_title,match_domain,similarity,mutual,zoa_sector,match_sector
0,Physical Violence Against Women,Physical Violence Against Women,irw_indikit_net,1.0,True,Gender-Based Violence,Gender-Based Violence Indicators
1,WASH Promoters' Knowledge and Skills,WASH Promoters' Knowledge and Skills,irw_indikit_net,1.0,True,Water,Hygiene Indicators
2,Hygiene Education at Schools,Hygiene Education at Schools,irw_indikit_net,1.0,True,Hygiene,Hygiene Indicators
3,Solid Waste Management,Solid Waste Management,irw_indikit_net,1.0,True,Sanitation,Solid Waste Management Indicators
4,Number of Selling Points,Number of Selling Points,irw_indikit_net,1.0,True,Market Development,Climate-Smart Energy Indicators
5,Number of Service Providers,Number of Service Providers,irw_indikit_net,1.0,True,Market Development,Sustainable Consumption and Production Indicators
6,Proportion of Surviving Seedlings,Proportion of Surviving Seedlings,irw_indikit_net,1.0,True,Agriculture and Natural Resource Management,Landscape Management Indicators
7,Shelters with Hazard Mitigation Measures,Shelters with Hazard Mitigation Measures,irw_indikit_net,1.0,True,Shelter and Disaster Risk Reduction,Shelter and NFI
8,Successfully Trained Teachers/ Trainers,Successfully Trained Teachers/ Trainers,irw_indikit_net,1.0,True,"Skills, Training and Employability",VET Quality and Relevance Indicators
9,Self-Help Groups - Functionality,Self-Help Groups - Functionality,irw_indikit_net,1.0,True,"Community Engagement, Participation & Advocacy",Social Protection and Inclusion


In [18]:
# 5. Free-text semantic search across both libraries
def search(query, top_k=8):
    q = model.encode([query], normalize_embeddings=True)
    scores = (embeddings @ q.T).ravel()
    idx = scores.argsort()[::-1][:top_k]
    result = corpus.loc[idx, ["domain", "title", "sector", "url"]].copy()
    result.insert(0, "score", scores[idx].round(3))
    return result.reset_index(drop=True)


search("resilience of smallholder farmers to drought")

,score,domain,title,sector,url
0,0.559,whh_indikit_net,#ms09 Climate Resilience - Households using th...,Environment,https://whh.indikit.net/indicator/5048-ms09-cl...
1,0.482,helvetas_indikit_net,WFC 1.10 People more resilient to the adverse ...,People,https://helvetas.indikit.net/indicator/61-wfc-...
2,0.476,tearfund_indikit_net,Improved climate adaptation and resilience,Environment and Climate Change,https://tearfund.indikit.net/indicator/5876-im...
3,0.476,tearfund_indikit_net,Water resources - climate resilience monitoring,"Water, sanitation and hygiene (WASH)",https://tearfund.indikit.net/indicator/5866-wa...
4,0.469,helvetas_indikit_net,WFC 1.7 Smallholders implementing climate-resi...,People,https://helvetas.indikit.net/indicator/70-wfc-...
5,0.469,tearfund_indikit_net,Adoption of Resilience Strengthening Measures,"Disaster Preparedness, Anticipation and Resili...",https://tearfund.indikit.net/indicator/3723-ad...
6,0.468,indicators_savethechildren_net,Uptake of diversified / climate-resilient live...,Diversified/climate-resilient livelihood options,https://indicators.savethechildren.net/indicat...
7,0.464,tearfund_indikit_net,Implementation of climate resilience projects,Environment and Climate Change,https://tearfund.indikit.net/indicator/5878-im...


In [20]:
search("water related deaths")

,score,domain,title,sector,url
0,0.619,irw_indikit_net,Reduced Mortality from Water-Borne Diseases,Global Results Framework,https://irw.indikit.net/indicator/5793-reduced...
1,0.545,irw_indikit_net,Mortality Rate due to Poor Water and Sanitatio...,Global Results Framework,https://irw.indikit.net/indicator/5745-mortali...
2,0.530,adra_indikit_net,Access to Drinking Water,Water Indicators,https://adra.indikit.net/indicator/51-access-t...
3,0.527,adra_indikit_net,Treatment of Drinking Water,Water Indicators,https://adra.indikit.net/indicator/59-treatmen...
4,0.527,www_indikit_net,Treatment of Drinking Water,Water Quality Indicators,https://www.indikit.net/indicator/59-treatment...
5,0.511,irw_indikit_net,Treatment of Drinking Water,Water Indicators,https://irw.indikit.net/indicator/4035-treatme...
6,0.482,irw_indikit_net,Access to Drinking Water,Water Indicators,https://irw.indikit.net/indicator/4029-access-...
7,0.470,zoa_indikit_net,Access to Drinking Water,Water,https://zoa.indikit.net/indicator/5121-access-...
